# Leer Data

**John González**

*23 de mayo de 2026*

**Objetivo**: Leer y guardar la data

In [1]:
# ! uv add kagglehub

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nachiketkamod/weather-dataset-us")

print("Path to dataset files:", path)

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 2.59G/2.59G [00:43<00:00, 63.2MB/s]

Extracting files...


Path to dataset files: /home/vscode/.cache/kagglehub/datasets/nachiketkamod/weather-dataset-us/versions/1


In [7]:
# Leer algunas lineas de path/combined_data_1.txt
NUM_LINES_TO_READ = 20

file_path = f"{path}/Weather Data (US).csv"

with open(file_path, "r", encoding="latin-1") as f:
    for i, line in enumerate(f, start=1):
        print(line.rstrip())
        if i >= NUM_LINES_TO_READ:
            break

ï»¿ID,DATE,TMAX,TMIN,EVAP,PRCP,Latitude,Longitude,Elevation
USS0012M13S,3/26/1997,,,,0,37.66,-112.74,2920
US1OKCV0021,4/18/2007,,,,241,35.1715,-97.4262,355.1
USC00355174,10/7/1999,,,,,42.0078,-121.3186,1410.3
US1MDHW0012,10/14/2015,,,,0,39.3387,-76.9468,166.1
US1TXHRS014,10/4/2021,,,,0,32.7061,-94.1683,53.6
USR0000AGOP,3/7/2011,-28,-200,,,64.2381,-145.2669,463.3
USC00201675,9/26/2012,200,100,,0,41.9622,-84.9925,299.9
USC00389039,12/3/2000,33,-11,,0,33.9,-80.5206,76.2
USC00230657,5/10/2007,267,139,,3,37.0539,-93.5756,399.3
USC00206510,8/2/2018,267,156,,28,45.3614,-84.9511,228
USC00236045,4/16/2008,150,6,,0,36.5869,-89.5325,92
USC00108535,7/11/2012,356,94,,0,42.6514,-111.5833,1780.6
USC00445150,1/5/1997,239,39,,0,38.3683,-78.2503,175.9
USW00013728,2/8/2021,106,-55,,0,36.5728,-79.335,168.2
US1TXED0025,9/1/2021,,,,0,29.9284,-100.207,697.1
US1RINW0003,7/29/2008,,,,0,41.5328,-71.3822,36.9
US1PALH0005,7/20/2016,,,,0,40.6474,-75.6505,143.9
US1TNDV0152,1/29/2020,,,,0,36.1124,-86.8117,191.4
US1T

In [8]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark = SparkSession.builder \
    .appName("Wheater") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()


path_proyecto = "/workspaces/bigdata/proyecto/datos"
os.makedirs(path_proyecto, exist_ok=True)
output_path = f"{path_proyecto}/Weather.parquet"




Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/26 02:47:11 WARN Utils: Your hostname, codespaces-5672ca, resolves to a loopback address: 127.0.0.1; using 10.0.15.46 instead (on interface eth0)
26/05/26 02:47:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/26 02:47:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [10]:
# Leer CSV con Spark y guardar como Parquet
csv_file = file_path

df = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .option("encoding", "iso-8859-1") \
    .csv(csv_file)

# Mostrar esquema y primeras filas
df.printSchema()
df.show(5)

# Guardar como Parquet (sobrescribe si existe)
df.write.mode("overwrite").parquet(output_path)
print("Guardado en:", output_path)

# Verificación: leer el parquet guardado
df_parquet = spark.read.parquet(output_path)
print("Registros en parquet:", df_parquet.count())
df_parquet.show(3)


root
 |-- ID: string (nullable = true)
 |-- DATE: string (nullable = true)
 |-- TMAX: integer (nullable = true)
 |-- TMIN: integer (nullable = true)
 |-- EVAP: integer (nullable = true)
 |-- PRCP: integer (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Elevation: double (nullable = true)

+-----------+----------+----+----+----+----+--------+---------+---------+
|         ID|      DATE|TMAX|TMIN|EVAP|PRCP|Latitude|Longitude|Elevation|
+-----------+----------+----+----+----+----+--------+---------+---------+
|USS0012M13S| 3/26/1997|NULL|NULL|NULL|   0|   37.66|  -112.74|   2920.0|
|US1OKCV0021| 4/18/2007|NULL|NULL|NULL| 241| 35.1715| -97.4262|    355.1|
|USC00355174| 10/7/1999|NULL|NULL|NULL|NULL| 42.0078|-121.3186|   1410.3|
|US1MDHW0012|10/14/2015|NULL|NULL|NULL|   0| 39.3387| -76.9468|    166.1|
|US1TXHRS014| 10/4/2021|NULL|NULL|NULL|   0| 32.7061| -94.1683|     53.6|
+-----------+----------+----+----+----+----+--------+---------

Guardado en: /workspaces/bigdata/proyecto/datos/Weather.parquet


Registros en parquet: 155840906
+-----------+---------+----+----+----+----+--------+---------+---------+
|         ID|     DATE|TMAX|TMIN|EVAP|PRCP|Latitude|Longitude|Elevation|
+-----------+---------+----+----+----+----+--------+---------+---------+
|US1MDMG0040|9/12/2014|NULL|NULL|NULL|   3| 39.0259| -77.0744|     92.0|
|US1WAKG0079|12/4/2008|NULL|NULL|NULL|   3| 47.6408|-122.4082|     93.3|
|US1WAKG0079|7/14/2014|NULL|NULL|NULL|   3| 47.6408|-122.4082|     93.3|
+-----------+---------+----+----+----+----+--------+---------+---------+
only showing top 3 rows
